# Assigment 3: SIAF Web Scraping


In this case you would be working on a web scrapping project with SIAF. We need you to obtain information about the expenditure budget that is used in a program focused on “LEARNING ACHIEVEMENTS OF REGULAR BASIC EDUCATION STUDENTS” code 0090, in the 26 regional governments of the country including the constitutional province of Callao. The following images would show how to select the information for a case, your job would be to put all this in an Excel, it should also be from 2014 to 2024.

Go to  pagehis : https://apps5.mineco.gob.pe/transparencia/Navegador/default.aspx?y=2014&ap=ActProyaons:



In [31]:
#import libraries
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from io import StringIO
import os
import time
import requests

In [14]:
# Settings WebDriver
driver = webdriver.Chrome()
url = "https://apps5.mineco.gob.pe/transparencia/Navegador/default.aspx?y=2014&ap=ActProy"
driver.get(url)
time.sleep(5)  # Esperar a que la página cargue completamente

In [15]:
driver.maximize_window()

### 1. You will select the government level and it will show you the next options:

In [16]:
#if we put the same code use in class it get ERROR
goverment_levels_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_BtnTipoGobierno"]' )
goverment_levels_button.click()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="ctl00_CPH1_BtnTipoGobierno"]"}
  (Session info: chrome=133.0.6943.142); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF6279FC6A5+28789]
	(No symbol) [0x00007FF627965B20]
	(No symbol) [0x00007FF6277F8F9A]
	(No symbol) [0x00007FF62784F346]
	(No symbol) [0x00007FF62784F57C]
	(No symbol) [0x00007FF6278A2B17]
	(No symbol) [0x00007FF62787736F]
	(No symbol) [0x00007FF62789F7E3]
	(No symbol) [0x00007FF627877103]
	(No symbol) [0x00007FF62783FFC0]
	(No symbol) [0x00007FF627841273]
	GetHandleVerifier [0x00007FF627D41AED+3458237]
	GetHandleVerifier [0x00007FF627D5829C+3550316]
	GetHandleVerifier [0x00007FF627D4DB9D+3507565]
	GetHandleVerifier [0x00007FF627AC2C6A+841274]
	(No symbol) [0x00007FF6279709EF]
	(No symbol) [0x00007FF62796CB34]
	(No symbol

**Note**: An "iframe", short for "inline frame", is an HTML element used to embed another web page within a parent page. In web scraping, it's important to identify iframes because they have their own separate DOM (Document Object Model). Elements within an iframe cannot be directly accessed from the main page's DOM. Therefore, to interact with or extract data from elements inside an iframe, one must first switch to the iframe's context. Failure to do so will result in an inability to locate and manipulate these elements, making understanding and handling iframes essential for successful scraping of complex web pages.

In [17]:
#Comprobar si tiene iframe 
frames = driver.find_elements( By.TAG_NAME, "frame" )
if frames:
    print( 'There are frames in this web page.' )
    for i, frame in enumerate( frames ):
        frame_html = frame.get_attribute( 'outerHTML' )
        print( f'HTML code of frame { i }:' )
        print( frame_html )
else:
    print('No frames found on the page.' )

There are frames in this web page.
HTML code of frame 0:
<frame name="frame0" id="frame0" src="Navegar.aspx?y=2014&amp;ap=ActProy" scrolling="yes">


In [18]:
#Once identified the frame/iframe we can use the switch to frame method
frame = driver.find_element( By.ID, "frame0" )
driver.switch_to.frame( frame )

In [19]:
#Now we can click the Government Levels button
goverment_levels_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_BtnTipoGobierno"]' )
goverment_levels_button.click()

### 2. Choose regional governments, and then choose "sheet"(pliego) in the option who spends? :



In [20]:
# Let's access some data from Regional Governments
regional_governments_level_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_RptData_ctl03_TD0"]/input' )
regional_governments_level_button.click()

time.sleep( 2 )

sector_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_BtnSector"]' )
sector_button.click()

time.sleep( 2 )

regional_goverments_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_RptData_ctl02_TD0"]/input' )
regional_goverments_button.click()

time.sleep( 2 )

pliego_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_BtnPliego"]' )
pliego_button.click()

### 3. Choose one region and then click in the budget category button:

In [27]:
# I choose 447: GOBIERNO REGIONAL DE HUANCAVELICA

time.sleep( 2 )
region_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_RptData_ctl08_TD0"]' )
region_button.click()

time.sleep( 2 )
budget_category_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_BtnProgramaPpto"]' )
budget_category_button.click()

### 4. Finally look for the code 0090 : " LEARNING ACHIEVEMENTS OF REGULAR BASIC EDUCATION STUDENTS".

In [28]:
education_button = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_RptData_ctl20_TD0"]/input' )
education_button.click()


In [32]:
table_element = driver.find_element( By.XPATH, '//*[@id="tr19"]' )
table_html    = table_element.get_attribute( 'outerHTML' )
table_html_io = StringIO( table_html )
table_df      = pd.read_html( table_html_io )[ 0 ]

ValueError: No tables found

### 5. The final excel needs to have the next columns :

In [33]:
# We are missing the titles (headers). We will find them by locating them through their TAG NAMES.

headers_elements_r0 = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row0"]' ).find_elements( By.TAG_NAME, 'td' )
headers_r0          = [ element.text for element in headers_elements_r0 ]

headers_elements_r1 = driver.find_element( By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row1"]' ).find_elements( By.TAG_NAME, 'td' )
headers_r1          = [ element.text for element in headers_elements_r1 ]

print( headers_r0, headers_r1, sep = '\n' )

[' ', 'Categoría Presupuestal', 'PIA', 'PIM', 'Certificación', 'Compromiso Anual', 'Ejecución', 'Avance % ']
['Atención de Compromiso Mensual ', 'Devengado ', 'Girado ']


In [ ]:
driver.quit()

In [34]:
headers_r0 = [ header for header in headers_r0 if header not in [ 'Ejecución', 'Avance % ' ] ]
headers = headers_r0 + headers_r1 + [ 'Avance %' ]
print( headers )

[' ', 'Categoría Presupuestal', 'PIA', 'PIM', 'Certificación', 'Compromiso Anual', 'Atención de Compromiso Mensual ', 'Devengado ', 'Girado ', 'Avance %']


In [ ]:
table_df.columns = headers
table_df = table_df.drop( table_df.columns[ 0 ], axis = 1 )
table_df.head( 5 )

EXPORT

In [ ]:
path_siaf          = '../data/Part_I/SIAF'
path_siaf_regional = path_siaf + '/regional'
os.makedirs( path_siaf_regional, exist_ok = True )

In [ ]:
table_path = path_siaf_regional + '/regional_table.xlsx'
table_df.to_excel( table_path, index = False )